In [32]:
import sys
print(sys.executable)
!{sys.executable} -m pip install fiftyone

C:\Users\alexw\AppData\Local\Programs\Python\Python310\python.exe


In [48]:
"""We're using FiftyOne as Bird_IndividualID's similarity calculation is on O(n^2) time
as Bird_IndividualID compares every image with all other images in a directory.
Since we have limited time (and it took 3000 seconds to calc 
the similarity of just 67 images with Bird_IndividualID)
we're going to approach this differently and use a tutorial from here
https://towardsdatascience.com/find-and-remove-duplicate-images-in-your-dataset-3e3ec818b978/
to calculate embeddings and the resulting similarity of images"""

import fiftyone as fo
dataset = fo.Dataset.from_dir(
    "E:/Datasets/Dataset_B/masked-split_train_val_by_day/train",
    fo.types.ImageClassificationDirectoryTree,
)

 100% |███████████████| 4115/4115 [3.9s elapsed, 0s remaining, 1.2K samples/s]       


In [49]:
#compute embeddings for our dataset
import fiftyone.zoo as foz
#using imagenet
model = foz.load_zoo_model("mobilenet-v2-imagenet-torch")
embeddings = dataset.compute_embeddings(model)

print(embeddings.shape)

 100% |███████████████| 4115/4115 [1.7m elapsed, 0s remaining, 37.5 samples/s]      
(4115, 1280)


In [50]:
from sklearn.metrics.pairwise import cosine_similarity

"""The N x N similarity matrix provides a value between 0 (low similarity) and 1 (identical) for each pair of your N samples."""
similarity_matrix = cosine_similarity(embeddings)

print(similarity_matrix.shape)

(4115, 4115)


In [51]:
"""all diagonal values are 1 since every image is identical to itself."""
print(similarity_matrix)

[[1.         0.51762854 0.68800656 ... 0.53542297 0.71019454 0.57623458]
 [0.51762854 1.         0.56971534 ... 0.88090117 0.59529345 0.55409345]
 [0.68800656 0.56971534 1.         ... 0.51437386 0.85263983 0.82147685]
 ...
 [0.53542297 0.88090117 0.51437386 ... 1.         0.59428436 0.52375785]
 [0.71019454 0.59529345 0.85263983 ... 0.59428436 1.         0.82088563]
 [0.57623458 0.55409345 0.82147685 ... 0.52375785 0.82088563 1.        ]]


In [52]:
import numpy as np

"""We can subtract by the identity matrix (N x N matrix with 1’s on the diagonal and 0’s elsewhere) 
in order to zero out the diagonal so those values don’t show up when we look for samples with maximum similarity."""

n = len(similarity_matrix)
similarity_matrix = similarity_matrix - np.identity(n)

In [53]:
id_map = [s.id for s in dataset.select_fields(["id"])]

for idx, sample in enumerate(dataset):
    max_similarity = similarity_matrix[idx].max()
    sample["max_similarity"] = max_similarity
    sample.save()

In [39]:
session = fo.launch_app(dataset, browser="default")

In [40]:
#I chose 0.97 as that appears to be where the 'tail' of our distribution begins for similarity
#as our data is all masked, similar pose, against a black background - our similarity skews higher than with a 'natural' dataset..
thresh = 0.95

In [41]:
from fiftyone import ViewField as F

view = dataset.match(F("max_similarity") > thresh)
print(view)

Dataset:     2025.08.16.16.32.34.862204
Media type:  image
Num samples: 1017
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    ground_truth:     fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
    max_similarity:   fiftyone.core.fields.FloatField
View stages:
    1. Match(filter={'$expr': {'$gt': [...]}})


In [42]:
#we now remove images that seem too similar according to our thresh value
samples_to_remove = set()
samples_to_keep = set()
for idx, sample in enumerate(dataset):
    if sample.id not in samples_to_remove:
        # Keep the first instance of two duplicates
        samples_to_keep.add(sample.id)
        
        dup_idxs = np.where(similarity_matrix[idx] > thresh)[0]
        for dup in dup_idxs:
            # We kept the first instance so remove all other duplicates
            samples_to_remove.add(id_map[dup])
        if len(dup_idxs) > 0:
            sample.tags.append("has_duplicates")
            sample.save()
        
    else:
        sample.tags.append("duplicate")
        sample.save()
        
# If you want to remove the samples from the dataset database (does not remove them locally) 
#dataset.delete_samples(list(samples_to_remove))

In [47]:
#I'm doing things from a local dataset that I'm later uploading to colab
#Ergo, this does local deletions
def count_files():
    file_count = 0
    for root, dirs, files in os.walk("E:/Datasets/Dataset_B/masked-split_train_val_by_day/train/"):
        file_count += len(files)
    return file_count

print("files before deletion: ", count_files())

for idx, sample in enumerate(dataset):
    if sample.id in samples_to_remove:
        os.remove(sample.filepath)


print("files after deletion: ", count_files())

files before deletion:  4625
files after deletion:  4115
